In [6]:
!pip install pandas numpy


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
!pip install openpyxl


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
import numpy as np

# -------- CARGA ( openpyxl para los .xlsx) --------
fact     = pd.read_csv('FACT_SALES.csv', dtype={'WEEK':'string','ITEM_CODE':'string'})
dim_cat  = pd.read_csv('DIM_CATEGORY.csv')
dim_prod = pd.read_excel('DIM_PRODUCT.xlsx')     
dim_seg  = pd.read_excel('DIM_SEGMENT.xlsx')
dim_cal  = pd.read_excel('DIM_CALENDAR.xlsx')

# -------- REVISIÓN RÁPIDA (shape, tipos, 3 filas) --------
for nombre, df in [('FACT',fact), ('CAT',dim_cat), ('PROD',dim_prod), ('SEG',dim_seg), ('CAL',dim_cal)]:
    print(f'\n== {nombre} ==', df.shape)
    print(df.dtypes.head())
    print(df.head(3))

# -------- LIMPIEZA BÁSICA --------
# estandarizar nombres con mayúsculas y espacios -> _
for df in [fact, dim_cat, dim_prod, dim_seg, dim_cal]:
    df.columns = (df.columns.str.strip().str.upper().str.replace(r'\s+','_', regex=True))

# números y fechas
for c in ['TOTAL_UNIT_SALES','TOTAL_VALUE_SALES','TOTAL_UNIT_AVG_WEEKLY_SALES']:
    if c in fact.columns:
        fact[c] = pd.to_numeric(fact[c], errors='coerce').fillna(0)

if 'DATE' in dim_cal.columns:
    dim_cal['DATE'] = pd.to_datetime(dim_cal['DATE'], errors='coerce', dayfirst=True)

# quitando duplicados razonables
keys_seg = [k for k in ['CATEGORY','FORMAT','ATTR1','ATTR2','ATTR3'] 
            if k in dim_seg.columns]
if keys_seg: 
    dim_seg_unique  = dim_seg.drop_duplicates(subset=keys_seg).copy() 
else: 
    dim_seg_unique = dim_seg.copy()                                                      #123456789
fact     = fact.drop_duplicates(subset=['WEEK','ITEM_CODE'])
dim_prod = dim_prod.drop_duplicates(subset=['ITEM'])
if {'ID_CATEGORY'}.issubset(dim_cat.columns):
    dim_cat = dim_cat.drop_duplicates(subset=['ID_CATEGORY'])
dim_cal  = dim_cal.drop_duplicates(subset=['WEEK'])



# -------- UNIONES --------
# A) Ventas + Producto (ITEM_CODE ↔ ITEM)
cons = fact.merge(dim_prod, left_on='ITEM_CODE', right_on='ITEM', how='left')

# B) Categoría (renombre para no chocar)
if {'ID_CATEGORY','CATEGORY'}.issubset(dim_cat.columns):
    dim_cat2 = dim_cat.rename(columns={'ID_CATEGORY':'CATEGORY_ID','CATEGORY':'CATEGORY_NAME'})
    if 'CATEGORY' in cons.columns:
        cons = cons.merge(dim_cat2, left_on='CATEGORY', right_on='CATEGORY_ID', how='left')

# C) Segmento por llaves comunes que existan
join_keys = [k for k in ['CATEGORY','FORMAT','ATTR1','ATTR2','ATTR3'] if k in cons.columns and k in dim_seg_unique.columns]
if join_keys:
    cons = cons.merge(dim_seg_unique[join_keys+['SEGMENT']], on=join_keys, how='left')

# D) Calendario por WEEK
if 'WEEK' in cons.columns and 'WEEK' in dim_cal.columns:
    cons = cons.merge(dim_cal[['WEEK','YEAR','MONTH','WEEK_NUMBER','DATE']], on='WEEK', how='left')

# -------- TRANSFORMACIONES SENCILLAS --------
cons = cons.rename(columns={'TOTAL_UNIT_SALES':'QTY', 'TOTAL_VALUE_SALES':'REVENUE'})
if {'QTY','REVENUE'}.issubset(cons.columns):
    cons['UNIT_PRICE'] = cons['REVENUE'] / cons['QTY'].replace(0, np.nan)

# -------- SELECCIÓN Y GUARDADO --------
final_cols = [c for c in [
    'WEEK','YEAR','MONTH','WEEK_NUMBER','DATE',
    'ITEM_CODE','ITEM','ITEM_DESCRIPTION','BRAND','MANUFACTURER',
    'CATEGORY','CATEGORY_ID','CATEGORY_NAME','FORMAT','ATTR1','ATTR2','ATTR3','SEGMENT',
    'REGION','QTY','REVENUE','TOTAL_UNIT_AVG_WEEKLY_SALES','UNIT_PRICE'
] if c in cons.columns]

df_final = cons[final_cols].copy()
df_final.to_csv('consolidado_ventas.csv', index=False, encoding='utf-8')
print('\n Listo:', df_final.shape, '-> consolidado_ventas.csv')
print(df_final.head(3))



== FACT == (122002, 6)
WEEK                           string[python]
ITEM_CODE                      string[python]
TOTAL_UNIT_SALES                      float64
TOTAL_VALUE_SALES                     float64
TOTAL_UNIT_AVG_WEEKLY_SALES           float64
dtype: object
    WEEK         ITEM_CODE  TOTAL_UNIT_SALES  TOTAL_VALUE_SALES  \
0  34-22  7501058792808BP2             0.006              0.139   
1  34-22     7501058715883             0.487            116.519   
2  34-22     7702626213774             1.391             68.453   

   TOTAL_UNIT_AVG_WEEKLY_SALES              REGION  
0                        1.000  TOTAL AUTOS AREA 5  
1                        2.916  TOTAL AUTOS AREA 5  
2                        5.171  TOTAL AUTOS AREA 5  

== CAT == (5, 2)
ID_CATEGORY     int64
CATEGORY       object
dtype: object
   ID_CATEGORY                        CATEGORY
0            1  FABRIC TREATMENT and SANIT\r\n
1            2                       AIR CARE 
2            3                    

In [7]:
print("FACT:", fact.shape, "FINAL:", df_final.shape)

unmatched_prod = fact[~fact['ITEM_CODE'].isin(dim_prod['ITEM'])]
unmatched_week = fact[~fact['WEEK'].isin(dim_cal['WEEK'])]
print("Sin match en PRODUCT:", len(unmatched_prod))
print("Sin match en CALENDAR:", len(unmatched_week))

print("ITEM duplicado en PROD:", dim_prod['ITEM'].duplicated().sum())
print("WEEK duplicado en CAL:", dim_cal['WEEK'].duplicated().sum())
print("ID_CATEGORY duplicado en CAT:", dim_cat['ID_CATEGORY'].duplicated().sum())
print("Pair (CATEGORY, FORMAT) duplicado en SEG:", dim_seg.duplicated(subset=['CATEGORY','FORMAT']).sum())

df_final.isna().sum().sort_values(ascending=False).head(10)

df_final[['QTY','REVENUE','TOTAL_UNIT_AVG_WEEKLY_SALES']].dtypes

print("Sum QTY vs FACT:", df_final['QTY'].sum(), fact['TOTAL_UNIT_SALES'].sum())
print("Sum REVENUE vs FACT:", df_final['REVENUE'].sum(), fact['TOTAL_VALUE_SALES'].sum())

if 'CATEGORY_NAME' in df_final.columns:
    df_final['CATEGORY_NAME'] = (df_final['CATEGORY_NAME']
                                 .astype(str).str.replace(r'\s+', ' ', regex=True).str.strip())
print: ('Duplicados en dim_segment por claves:', dim_seg.duplicated(subset=keys_seg).sum())
print: ('Duplicados en dim_segment_unique por claves:', dim_seg_unique.duplicated(subset=keys_seg).sum())

FACT: (20990, 6) FINAL: (20990, 23)
Sin match en PRODUCT: 0
Sin match en CALENDAR: 0
ITEM duplicado en PROD: 0
WEEK duplicado en CAL: 0
ID_CATEGORY duplicado en CAT: 0
Pair (CATEGORY, FORMAT) duplicado en SEG: 48
Sum QTY vs FACT: 52877.22 52877.22
Sum REVENUE vs FACT: 1447043.921 1447043.921
